# Extract FIDE Calculation Pages for Fixed 1,000-Player Sample

**Purpose:**  
This notebook extracts FIDE rating-calculation page data for the fixed 1,000-player sample used in the capstone project. The extracted calculation-page data provides opponent-level enrichment variables such as opponent rating, score, tournament exposure, rating change, and K-adjusted rating change.


**Main outputs:**  
- Player-level calculation-page extraction results  
- Opponent-game level records  
- Combined calculation-page CSV files used later for feature engineering and modelling


### Fide historical calculation for sample playes

In [1]:
# ============================================================
# Step 1: Import required libraries
# Purpose:
# - Load packages for file handling, data processing, web access/parsing,
#   waiting/retry logic, and progress monitoring.
# ============================================================

import pandas as pd
import numpy as np
import re
import time
from pathlib import Path
from bs4 import BeautifulSoup
from io import StringIO
from playwright.async_api import async_playwright

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", None)

## 1. Setup, imports, and path configuration

This section loads required libraries and defines paths for reading the fixed 1,000-player sample and saving extracted FIDE calculation-page outputs.


#### Test extraction from Fide website

In [5]:
# ============================================================
# Step 3: Create output folders
# Purpose:
# - Ensure extraction output directories exist before saving HTML/CSV files.
# ============================================================


fide_id = "25977890"   # Amartya
rating_type = 0        # 0 = Standard, 1 = Rapid, 2 = Blitz

start_period = "2024-01-01"
end_period = "2026-05-01"

periods = pd.date_range(start_period, end_period, freq="MS")

output_dir = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw" / "fide_history" / "amartya_calculations"
output_dir.mkdir(parents=True, exist_ok=True)

print("Months to process:", len(periods))
print(periods)

Months to process: 29
DatetimeIndex(['2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01', '2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01', '2025-01-01', '2025-02-01', '2025-03-01', '2025-04-01', '2025-05-01', '2025-06-01', '2025-07-01', '2025-08-01', '2025-09-01', '2025-10-01', '2025-11-01', '2025-12-01', '2026-01-01', '2026-02-01', '2026-03-01', '2026-04-01', '2026-05-01'], dtype='datetime64[ns]', freq='MS')


In [7]:
# ============================================================
# Step 4: Load the fixed player sample
# Purpose:
# - Read the 1,000-player sample that will be used for calculation-page extraction.
# - Keep FIDE IDs in a stable format for URL construction.
# ============================================================

def extract_event_metadata_from_text(text_lines):
    """
    Finds tournament blocks before each Rc/Ro/w/n/chg/K/K*chg table.
    Expected pattern:
    tournament_name
    CITY FED
    start_date
    end_date
    Rc
    Ro
    ...
    """
    events = []

    for i, line in enumerate(text_lines):
        if line == "Rc":
            if i >= 4:
                event_name = text_lines[i - 4]
                location_fed = text_lines[i - 3]
                start_date = text_lines[i - 2]
                end_date = text_lines[i - 1]

                # Validate dates lightly
                if re.match(r"^\d{4}-\d{2}-\d{2}$", start_date) and re.match(r"^\d{4}-\d{2}-\d{2}$", end_date):
                    events.append({
                        "tournament_name": event_name,
                        "event_location_fed": location_fed,
                        "event_start_date": start_date,
                        "event_end_date": end_date
                    })

    return events


def clean_fide_calculation_table(table, event_meta, fide_id, period, rating_type=0):
    """
    Cleans one FIDE calculation table from rendered page.
    Output = one row per opponent game.
    """

    df = table.copy()

    # Keep first 10 useful columns only
    df = df.iloc[:, :10].copy()

    # Need at least summary + opponent rows
    if df.shape[0] < 4:
        return pd.DataFrame()

    # Row 1 usually contains event summary
    summary = df.iloc[1]

    event_rc = pd.to_numeric(summary.iloc[0], errors="coerce")
    event_ro = pd.to_numeric(summary.iloc[1], errors="coerce")
    event_score = pd.to_numeric(summary.iloc[5], errors="coerce")
    event_games = pd.to_numeric(summary.iloc[6], errors="coerce")
    event_chg = pd.to_numeric(summary.iloc[7], errors="coerce")
    event_k = pd.to_numeric(summary.iloc[8], errors="coerce")
    event_kchg = pd.to_numeric(summary.iloc[9], errors="coerce")

    # Opponent rows generally start after header, summary, blank row
    opponent_df = df.iloc[3:].copy()

    opponent_df.columns = [
        "opponent_name",
        "opponent_title_1",
        "opponent_title_2",
        "opponent_rating_raw",
        "opponent_fed",
        "score",
        "n",
        "chg",
        "k",
        "k_chg"
    ]

    # Remove empty rows
    opponent_df = opponent_df[opponent_df["opponent_name"].notna()].copy()

    # Remove footnote rows
    opponent_df = opponent_df[
        ~opponent_df["opponent_name"].astype(str).str.contains(
            "Rating difference", case=False, na=False
        )
    ].copy()

    # Federation must be valid 3-letter code
    opponent_df["opponent_fed"] = opponent_df["opponent_fed"].astype(str).str.strip()

    opponent_df = opponent_df[
        opponent_df["opponent_fed"].str.match(r"^[A-Z]{3}$", na=False)
    ].copy()

    if opponent_df.empty:
        return pd.DataFrame()

    # Combine title columns
    opponent_df["opponent_title"] = (
        opponent_df[["opponent_title_1", "opponent_title_2"]]
        .astype(str)
        .replace("nan", "", regex=False)
        .agg(" ".join, axis=1)
        .str.strip()
    )

    opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan).infer_objects(copy=False)

    # Flag 400+ rating difference note
    opponent_df["rating_difference_over_400_flag"] = (
        opponent_df["opponent_rating_raw"]
        .astype(str)
        .str.contains(r"\*", regex=True, na=False)
    )

    # Clean opponent rating
    opponent_df["opponent_rating"] = (
        opponent_df["opponent_rating_raw"]
        .astype(str)
        .str.replace("*", "", regex=False)
        .str.strip()
    )

    opponent_df["opponent_rating"] = pd.to_numeric(
        opponent_df["opponent_rating"], errors="coerce"
    )

    # Numeric columns
    for col in ["score", "n", "chg", "k", "k_chg"]:
        opponent_df[col] = pd.to_numeric(opponent_df[col], errors="coerce")

    # Add player/month/event fields
    opponent_df["fide_id"] = str(fide_id)
    opponent_df["period"] = period
    opponent_df["rating_type"] = rating_type

    opponent_df["tournament_name"] = event_meta.get("tournament_name")
    opponent_df["event_location_fed"] = event_meta.get("event_location_fed")
    opponent_df["event_start_date"] = event_meta.get("event_start_date")
    opponent_df["event_end_date"] = event_meta.get("event_end_date")

    opponent_df["event_rc"] = event_rc
    opponent_df["event_ro"] = event_ro
    opponent_df["event_score"] = event_score
    opponent_df["event_games"] = event_games
    opponent_df["event_chg"] = event_chg
    opponent_df["event_k"] = event_k
    opponent_df["event_k_chg"] = event_kchg

    keep_cols = [
        "fide_id", "period", "rating_type",
        "tournament_name", "event_location_fed",
        "event_start_date", "event_end_date",
        "event_rc", "event_ro", "event_score", "event_games",
        "event_chg", "event_k", "event_k_chg",
        "opponent_name", "opponent_title", "opponent_rating",
        "rating_difference_over_400_flag",
        "opponent_fed", "score", "n", "chg", "k", "k_chg"
    ]

    return opponent_df[keep_cols].reset_index(drop=True)

## 2. Load fixed 1,000-player sample

This section loads the sample of players for whom calculation-page data will be extracted. The sample should remain fixed so downstream modelling results remain stable.


In [10]:
# ============================================================
# Step 5: Inspect sample structure
# Purpose:
# - Verify player count, key columns, and sample rows before extraction begins.
# ============================================================

all_months_clean = []
month_status = []

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    page = await browser.new_page()

    for period in periods:
        period_str = period.strftime("%Y-%m-%d")

        url = (
            "https://ratings.fide.com/calculations.phtml"
            f"?id_number={fide_id}&period={period_str}&rating={rating_type}"
        )

        print("\nProcessing:", period_str)
        print(url)

        try:
            await page.goto(url, wait_until="networkidle", timeout=60000)
            await page.wait_for_timeout(3000)

            html = await page.content()

            # Save rendered HTML for audit/debug
            html_path = output_dir / f"rendered_{period_str}_standard.html"
            html_path.write_text(html, encoding="utf-8")

            # Optional screenshot for months with data/debugging
            # screenshot_path = output_dir / f"rendered_{period_str}_standard.png"
            # await page.screenshot(path=str(screenshot_path), full_page=True)

            soup = BeautifulSoup(html, "html.parser")

            text_lines = [
                line.strip()
                for line in soup.get_text("\n", strip=True).split("\n")
                if line.strip()
            ]

            try:
                tables = pd.read_html(StringIO(html))
            except ValueError:
                tables = []

            events = extract_event_metadata_from_text(text_lines)

            print("Tables found:", len(tables))
            print("Events found:", len(events))

            clean_tables = []

            for i, table in enumerate(tables):
                event_meta = events[i] if i < len(events) else {}

                clean_one = clean_fide_calculation_table(
                    table=table,
                    event_meta=event_meta,
                    fide_id=fide_id,
                    period=period_str,
                    rating_type=rating_type
                )

                if not clean_one.empty:
                    clean_tables.append(clean_one)

            if clean_tables:
                month_clean = pd.concat(clean_tables, ignore_index=True)

                all_months_clean.append(month_clean)

                print("Clean opponent rows:", month_clean.shape[0])
                print(month_clean[[
                    "period", "tournament_name", "opponent_name",
                    "opponent_rating", "opponent_fed", "score", "chg", "k_chg"
                ]].head(10).to_string(index=False))

                month_status.append({
                    "period": period_str,
                    "url": url,
                    "tables_found": len(tables),
                    "events_found": len(events),
                    "clean_rows": month_clean.shape[0],
                    "status": "data_found"
                })

            else:
                print("No clean rows for this month.")

                month_status.append({
                    "period": period_str,
                    "url": url,
                    "tables_found": len(tables),
                    "events_found": len(events),
                    "clean_rows": 0,
                    "status": "no_data_or_no_clean_rows"
                })

            await page.wait_for_timeout(1500)

        except Exception as e:
            print("Failed:", period_str, repr(e))

            month_status.append({
                "period": period_str,
                "url": url,
                "tables_found": None,
                "events_found": None,
                "clean_rows": 0,
                "status": "failed",
                "error": repr(e)
            })

    await browser.close()


Processing: 2024-01-01
https://ratings.fide.com/calculations.phtml?id_number=25977890&period=2024-01-01&rating=0
Tables found: 3
Events found: 3
Clean opponent rows: 24
    period                                                     tournament_name    opponent_name  opponent_rating opponent_fed  score   chg  k_chg
2024-01-01 Bijnor Open FIDE Rated Chess Tournament - UP Booster Series (Vol I)  Kshitika, Tyagi           1010.0          IND    1.0  0.10   2.90
2024-01-01 Bijnor Open FIDE Rated Chess Tournament - UP Booster Series (Vol I)     Dhruv, Gupta           1191.0          IND    1.0  0.26   7.54
2024-01-01 Bijnor Open FIDE Rated Chess Tournament - UP Booster Series (Vol I)   Nikhil, Chopra           1197.0          IND    1.0  0.27   7.83
2024-01-01 Bijnor Open FIDE Rated Chess Tournament - UP Booster Series (Vol I)    Vatsal, Makol           1477.0          IND    0.5  0.14   4.06
2024-01-01 Bijnor Open FIDE Rated Chess Tournament - UP Booster Series (Vol I)      Rahul, Suri     

/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_49366/3210746287.py:113: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan).infer_objects(copy=False)
/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_49366/3210746287.py:113: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan).infer_objects(copy=False)
/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_4


Processing: 2024-02-01
https://ratings.fide.com/calculations.phtml?id_number=25977890&period=2024-02-01&rating=0
Tables found: 1
Events found: 1
Clean opponent rows: 8
    period                                            tournament_name      opponent_name  opponent_rating opponent_fed  score   chg  k_chg
2024-02-01 2nd Matrix International Open FIDE Rating Chess Tournament     Kavyansh, Jain           1125.0          IND    1.0  0.11    4.4
2024-02-01 2nd Matrix International Open FIDE Rating Chess Tournament Anant, Singh Dagar           1217.0          IND    1.0  0.19    7.6
2024-02-01 2nd Matrix International Open FIDE Rating Chess Tournament       Aansh, Gupta           1991.0          IND    1.0  0.97   38.8
2024-02-01 2nd Matrix International Open FIDE Rating Chess Tournament    Pradeep, Tiwari           1809.0          IND    0.5  0.38   15.2
2024-02-01 2nd Matrix International Open FIDE Rating Chess Tournament     Ashok, Gajwani           1853.0          IND    0.0 -0.09   -3

/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_49366/3210746287.py:113: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan).infer_objects(copy=False)



Processing: 2024-03-01
https://ratings.fide.com/calculations.phtml?id_number=25977890&period=2024-03-01&rating=0
Tables found: 2
Events found: 2
Clean opponent rows: 16
    period                                                 tournament_name     opponent_name  opponent_rating opponent_fed  score   chg  k_chg
2024-03-01               National School Chess Championship 2023-24 (U13O)     Reddi, Satwik           1114.0          IND    1.0  0.08    3.2
2024-03-01               National School Chess Championship 2023-24 (U13O)     Nikhil, Kumar           1284.0          IND    0.5 -0.29  -11.6
2024-03-01               National School Chess Championship 2023-24 (U13O)  Tejas, Shandilya           1216.0          IND    1.0  0.14    5.6
2024-03-01               National School Chess Championship 2023-24 (U13O)  Swapnabha, Neogi           1239.0          IND    0.5 -0.34  -13.6
2024-03-01               National School Chess Championship 2023-24 (U13O)  Devansh, Rathore           1206.0      

...


Processing: 2025-09-01
https://ratings.fide.com/calculations.phtml?id_number=25977890&period=2025-09-01&rating=0
Tables found: 1
Events found: 1
Clean opponent rows: 9
    period                                                                   tournament_name          opponent_name  opponent_rating opponent_fed  score   chg  k_chg
2025-09-01 SHRI DHANPAT RAI SACHDEVA MEMORIAL OPEN INTERNATIONAL FIDE RATED CHESS TOURNAMENT          Dheekshika, G           1542.0          IND    1.0  0.13    5.2
2025-09-01 SHRI DHANPAT RAI SACHDEVA MEMORIAL OPEN INTERNATIONAL FIDE RATED CHESS TOURNAMENT         Sharma, Pankaj           1661.0          IND    0.5 -0.25  -10.0
2025-09-01 SHRI DHANPAT RAI SACHDEVA MEMORIAL OPEN INTERNATIONAL FIDE RATED CHESS TOURNAMENT            Rahul, Suri           1645.0          IND    1.0  0.23    9.2
2025-09-01 SHRI DHANPAT RAI SACHDEVA MEMORIAL OPEN INTERNATIONAL FIDE RATED CHESS TOURNAMENT        Rituraj, Tamuli           1748.0          IND    1.0  0.35   14.0
2

/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_49366/3210746287.py:113: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan).infer_objects(copy=False)



Processing: 2026-01-01
https://ratings.fide.com/calculations.phtml?id_number=25977890&period=2026-01-01&rating=0
Tables found: 1
Events found: 1
Clean opponent rows: 9
    period                                                 tournament_name    opponent_name  opponent_rating opponent_fed  score   chg  k_chg
2026-01-01 1st Arunachal Pradesh International Chess Festival 2025 (Cat-B)      Tania, Arki           1541.0          IND    1.0  0.09    3.6
2026-01-01 1st Arunachal Pradesh International Chess Festival 2025 (Cat-B)  Abhibhav, Kumar           1724.0          IND    0.5 -0.27  -10.8
2026-01-01 1st Arunachal Pradesh International Chess Festival 2025 (Cat-B)     Talin, Nimpu           1768.0          IND    1.0  0.28   11.2
2026-01-01 1st Arunachal Pradesh International Chess Festival 2025 (Cat-B)     Saikat, Nath           1798.0          IND    0.5 -0.18   -7.2
2026-01-01 1st Arunachal Pradesh International Chess Festival 2025 (Cat-B)      C,r, Ritvik           1823.0          IND

/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_49366/3210746287.py:113: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan).infer_objects(copy=False)



Processing: 2026-02-01
https://ratings.fide.com/calculations.phtml?id_number=25977890&period=2026-02-01&rating=0
Tables found: 3
Events found: 3
Clean opponent rows: 26
    period                                          tournament_name                   opponent_name  opponent_rating opponent_fed  score   chg  k_chg
2026-02-01              VI Open Internacional de Ajedrez Dama Negra           Santos Anton, Armando           1532.0          ESP    1.0  0.08   2.08
2026-02-01              VI Open Internacional de Ajedrez Dama Negra                  Doman, Richard           1652.0          ENG    1.0  0.16   4.16
2026-02-01              VI Open Internacional de Ajedrez Dama Negra                  Prabhu, Aadith           1831.0          SCO    1.0  0.36   9.36
2026-02-01              VI Open Internacional de Ajedrez Dama Negra          Peixoto, Sergio Dantas           1778.0          BRA    1.0  0.29   7.54
2026-02-01              VI Open Internacional de Ajedrez Dama Negra           Tr

In [11]:
# ============================================================
# Step 6: Define months / periods for extraction
# Purpose:
# - Specify rating periods for which FIDE calculation pages will be checked.
# ============================================================

# Combine and save
month_status_df = pd.DataFrame(month_status)

status_path = output_dir / "amartya_month_status.csv"
month_status_df.to_csv(status_path, index=False)

print("Saved status:", status_path)
print(month_status_df.to_string(index=False))

Saved status: /Users/arunkumar/Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess/data/raw/fide_history/amartya_calculations/amartya_month_status.csv
    period                                                                                       url  tables_found  events_found  clean_rows                   status
2024-01-01 https://ratings.fide.com/calculations.phtml?id_number=25977890&period=2024-01-01&rating=0             3             3          24               data_found
2024-02-01 https://ratings.fide.com/calculations.phtml?id_number=25977890&period=2024-02-01&rating=0             1             1           8               data_found
2024-03-01 https://ratings.fide.com/calculations.phtml?id_number=25977890&period=2024-03-01&rating=0             2             2          16               data_found
2024-04-01 https://ratings.fide.com/calculations.phtml?id_number=25977890&period=2024-04-01&rating=0             1             1           8     

In [12]:
# ============================================================
# Step 7: Build calculation-page URL helper
# Purpose:
# - Create a reusable function or pattern for constructing FIDE calculation URLs.
# ============================================================

#
if all_months_clean:
    amartya_all_calc = pd.concat(all_months_clean, ignore_index=True)

    output_csv = output_dir / "amartya_all_months_standard_calculations_clean.csv"
    amartya_all_calc.to_csv(output_csv, index=False)

    print("Saved clean calculations:", output_csv)
    print("Shape:", amartya_all_calc.shape)
    print(amartya_all_calc.head(30).to_string(index=False))
else:
    amartya_all_calc = pd.DataFrame()
    print("No clean calculation data found for any month.")

Saved clean calculations: /Users/arunkumar/Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess/data/raw/fide_history/amartya_calculations/amartya_all_months_standard_calculations_clean.csv
Shape: (321, 24)
 fide_id     period  rating_type                                                                 tournament_name event_location_fed event_start_date event_end_date  event_rc  event_ro  event_score  event_games  event_chg  event_k  event_k_chg              opponent_name opponent_title  opponent_rating  rating_difference_over_400_flag opponent_fed  score  n   chg  k  k_chg
25977890 2024-01-01            0             Bijnor Open FIDE Rated Chess Tournament - UP Booster Series (Vol I)         Bijnor IND       2023-11-30     2023-12-03      1255      1376          5.0            6       1.15       29        33.35            Kshitika, Tyagi            NaN           1010.0                            False          IND    1.0  1  0.10 29   2.90
259778

## 3. Build FIDE calculation-page URLs

This section constructs FIDE rating-calculation URLs for player-month combinations. These URLs are used to access opponent-level rating-calculation records.


In [7]:
# ============================================================
# Step 8: Test URL generation
# Purpose:
# - Confirm that FIDE ID and month values produce valid calculation-page links.
# ============================================================

#Check monthly summary
if not amartya_all_calc.empty:
    monthly_summary = (
        amartya_all_calc
        .groupby("period", as_index=False)
        .agg(
            tournaments=("tournament_name", "nunique"),
            games=("opponent_name", "count"),
            avg_opponent_rating=("opponent_rating", "mean"),
            total_score=("score", "sum"),
            total_rating_change=("k_chg", "sum"),
            avg_chg=("chg", "mean")
        )
    )

    print(monthly_summary.to_string(index=False))

    period  tournaments  games  avg_opponent_rating  total_score  total_rating_change   avg_chg
2024-01-01            3     24          1339.625000         16.5         9.628000e+01  0.138333
2024-02-01            1      8          1461.500000          5.5         4.680000e+01  0.146250
2024-03-01            2     16          1479.375000          9.5         3.640000e+01  0.056875
2024-04-01            1      8          1658.500000          4.5        -1.440000e+01 -0.045000
2024-05-01            2     17          1775.941176          7.5        -8.000000e-01 -0.001176
2024-06-01            1      8          1539.250000          6.0         5.600000e+00  0.017500
2024-07-01            2     17          1616.647059         11.5         2.440000e+01  0.035882
2024-08-01            1      7          1482.857143          4.0        -6.320000e+01 -0.225714
2024-09-01            2     19          1790.368421          9.0         6.984000e+01  0.102105
2024-10-01            1     11          

In [13]:
# ============================================================
# Step 9: Define extraction/parsing helper functions
# Purpose:
# - Encapsulate logic for reading FIDE calculation pages and converting tables
#   into structured DataFrames.
# ============================================================

#Check tournament summary
if not amartya_all_calc.empty:
    tournament_summary = (
        amartya_all_calc
        .groupby(["period", "tournament_name"], as_index=False)
        .agg(
            event_location_fed=("event_location_fed", "first"),
            event_start_date=("event_start_date", "first"),
            event_end_date=("event_end_date", "first"),
            event_rc=("event_rc", "first"),
            event_ro=("event_ro", "first"),
            event_score=("event_score", "first"),
            event_games=("event_games", "first"),
            event_k_chg=("event_k_chg", "first"),
            games=("opponent_name", "count"),
            avg_opponent_rating=("opponent_rating", "mean"),
            score_sum=("score", "sum"),
            k_chg_sum=("k_chg", "sum")
        )
    )

    print(tournament_summary.to_string(index=False))

    period                                                                   tournament_name            event_location_fed event_start_date event_end_date  event_rc  event_ro  event_score  event_games  event_k_chg  games  avg_opponent_rating  score_sum     k_chg_sum
2024-01-01                                    36th National Under-13 Chess Championship-2023                 Hyderabad IND       2023-12-04     2023-12-10      1464      1376          6.0           10        58.87     10          1464.200000        6.0  5.887000e+01
2024-01-01               Bijnor Open FIDE Rated Chess Tournament - UP Booster Series (Vol I)                    Bijnor IND       2023-11-30     2023-12-03      1255      1376          5.0            6        33.35      6          1254.666667        5.0  3.335000e+01
2024-01-01   Winter Cup Classical Below 1600 FIDE Rating Lakecity Open Chess Tournament 2023                   Udaipur IND       2023-12-22     2023-12-24      1248      1376          5.5            

In [14]:
# ============================================================
# Step 10: Test extraction on a small subset
# Purpose:
# - Run extraction for a small number of players before processing the full sample.
# - This helps identify layout, missing-page, or parsing issues early.
# ============================================================

tournament_view = tournament_summary.copy()

# Shorten long tournament names for display only
tournament_view["tournament_name_short"] = (
    tournament_view["tournament_name"]
    .astype(str)
    .str.slice(0, 55)
)

cols = [
    "period",
    "tournament_name_short",
    "event_location_fed",
    "event_start_date",
    "event_end_date",
    "event_ro",
    "event_rc",
    "event_score",
    "event_games",
    "avg_opponent_rating",
    "k_chg_sum"
]

display(tournament_view[cols])

,period,tournament_name_short,event_location_fed,event_start_date,event_end_date,event_ro,event_rc,event_score,event_games,avg_opponent_rating,k_chg_sum
0,2024-01-01,36th National Under-13 Chess Championship-2023,Hyderabad IND,2023-12-04,2023-12-10,1376,1464,6.0,10,1464.200000,5.887000e+01
1,2024-01-01,Bijnor Open FIDE Rated Chess Tournament - UP Booster Se,Bijnor IND,2023-11-30,2023-12-03,1376,1255,5.0,6,1254.666667,3.335000e+01
2,2024-01-01,Winter Cup Classical Below 1600 FIDE Rating Lakecity Op,Udaipur IND,2023-12-22,2023-12-24,1376,1248,5.5,8,1247.625000,4.060000e+00
3,2024-02-01,2nd Matrix International Open FIDE Rating Chess Tournam,Delhi IND,2024-01-03,2024-01-07,1472,1462,5.5,8,1461.500000,4.680000e+01
4,2024-03-01,11th National Amateur Chess Championship 2023-24 for Be,Jaipur IND,2024-02-13,2024-02-17,1519,1630,4.0,8,1629.625000,4.440000e+01
5,2024-03-01,National School Chess Championship 2023-24 (U13O),Patna IND,2024-02-06,2024-02-10,1519,1329,5.5,8,1329.125000,-8.000000e+00
6,2024-04-01,Bijnor Open FIDE Rated Chess Tournament UP Booster Ser,Moradabad IND,2024-03-17,2024-03-21,1733,1659,4.5,8,1658.500000,-1.440000e+01
7,2024-05-01,3rd Matrix Cup International Open Fide Rated Chess Tour,Delhi IND,2024-04-06,2024-04-10,1719,1642,4.5,8,1641.875000,-1.000000e+01
8,2024-05-01,43rd National Team Chess Championship 2023-24,Dharmasala IND,2024-03-29,2024-04-04,1733,1895,3.0,9,1895.111111,9.200000e+00
9,2024-06-01,Delhi State Chess Championship-2024,Delhi IND,2024-05-26,2024-05-28,1718,1539,6.0,8,1539.250000,5.600000e+00


## 4. Extract / scrape calculation-page records

This section fetches or parses the FIDE calculation-page content and converts the visible calculation tables into structured rows.


#### Extraction for 1000 sample players

#### 1. Load the 1,000-player sample

Use whichever sample you selected, representative or balanced.

In [16]:
# ============================================================
# Step 13: Combine extracted rows
# Purpose:
# - Concatenate all extracted player/month tables into a single DataFrame.
# ============================================================

import pandas as pd
import numpy as np
import re
import time
import math
from pathlib import Path
from bs4 import BeautifulSoup
from io import StringIO
from playwright.async_api import async_playwright

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", 80)

In [17]:
# ============================================================
# Step 14: Clean calculation-page rows
# Purpose:
# - Remove blank, invalid, duplicate, or non-game rows.
# - Standardize numeric columns for modelling.
# ============================================================

#sample_path = Path.home() / "Downloads" / "fide_history" / "indian_youth_1000_balanced_sample.csv"
# Or use representative sample:
# sample_path = Path.home() / "Downloads" / "fide_history" / "indian_youth_1000_player_features.csv"
player_sample_1000_output_path = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw" / "player_sample_1000.csv"

player_sample = pd.read_csv(player_sample_1000_output_path)

player_sample.columns = player_sample.columns.str.strip()
player_sample["ID Number"] = player_sample["ID Number"].astype(str).str.strip()

player_ids = player_sample["ID Number"].dropna().drop_duplicates().tolist()

print("Players:", len(player_ids))
print(player_ids[:10])

Players: 1000
['558016627', '531075452', '537082728', '25789783', '45056404', '531047971', '48773603', '558008276', '88138674', '48724211']


## 5. Clean extracted opponent-game records

This section removes invalid rows, blank rows, footnotes, and non-game rows, then standardizes ratings, scores, K-factors, and rating-change fields.


#### 2. Set extraction period

For your feature window, use Apr 2024 to Apr 2025.

In [18]:
# ============================================================
# Step 16: Create coverage indicators
# Purpose:
# - Count players with and without usable calculation-page data.
# - These counts support the report's enrichment coverage summary.
# ============================================================

rating_type = 0  # 0 = Standard, 1 = Rapid, 2 = Blitz

start_period = "2024-05-01"
end_period = "2025-04-01"

periods = pd.date_range(start_period, end_period, freq="MS")

print("Months:", len(periods))
print([p.strftime("%Y-%m-%d") for p in periods])

Months: 12
['2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01', '2025-01-01', '2025-02-01', '2025-03-01', '2025-04-01']


#### 3. Output folders

## 6. Combine player-level extraction outputs

This section combines extracted calculation-page records across players and months into one structured opponent-game dataset.


In [19]:
# ============================================================
# Step 18: Save combined cleaned calculation-page dataset
# Purpose:
# - Export the cleaned opponent-game level dataset for feature engineering.
# ============================================================

#base_output_dir = Path.home() / "Downloads" / "fide_history" / "fide_calculation_pages"
base_output_dir = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw" / "fide_history" / "fide_calculation_pages"

raw_html_dir = base_output_dir / "raw_html"
clean_dir = base_output_dir / "clean_player_months"
log_dir = base_output_dir / "logs"

raw_html_dir.mkdir(parents=True, exist_ok=True)
clean_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

print(base_output_dir)

/Users/arunkumar/Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess/data/raw/fide_history/fide_calculation_pages


#### 4. Helper functions

In [20]:
# ============================================================
# Step 20: Final extraction checks
# Purpose:
# - Print final coverage and data-quality checks before using the file downstream.
# ============================================================

def extract_event_metadata_from_text(text_lines):
    events = []

    for i, line in enumerate(text_lines):
        if line == "Rc":
            if i >= 4:
                event_name = text_lines[i - 4]
                location_fed = text_lines[i - 3]
                start_date = text_lines[i - 2]
                end_date = text_lines[i - 1]

                if (
                    re.match(r"^\d{4}-\d{2}-\d{2}$", start_date)
                    and re.match(r"^\d{4}-\d{2}-\d{2}$", end_date)
                ):
                    events.append({
                        "tournament_name": event_name,
                        "event_location_fed": location_fed,
                        "event_start_date": start_date,
                        "event_end_date": end_date
                    })

    return events


def clean_fide_calculation_table(table, event_meta, fide_id, period, rating_type=0):
    df = table.copy()

    if df.shape[1] < 10 or df.shape[0] < 4:
        return pd.DataFrame()

    df = df.iloc[:, :10].copy()

    summary = df.iloc[1]

    event_rc = pd.to_numeric(summary.iloc[0], errors="coerce")
    event_ro = pd.to_numeric(summary.iloc[1], errors="coerce")
    event_score = pd.to_numeric(summary.iloc[5], errors="coerce")
    event_games = pd.to_numeric(summary.iloc[6], errors="coerce")
    event_chg = pd.to_numeric(summary.iloc[7], errors="coerce")
    event_k = pd.to_numeric(summary.iloc[8], errors="coerce")
    event_kchg = pd.to_numeric(summary.iloc[9], errors="coerce")

    opponent_df = df.iloc[3:].copy()

    opponent_df.columns = [
        "opponent_name",
        "opponent_title_1",
        "opponent_title_2",
        "opponent_rating_raw",
        "opponent_fed",
        "score",
        "n",
        "chg",
        "k",
        "k_chg"
    ]

    opponent_df = opponent_df[opponent_df["opponent_name"].notna()].copy()

    opponent_df = opponent_df[
        ~opponent_df["opponent_name"].astype(str).str.contains(
            "Rating difference", case=False, na=False
        )
    ].copy()

    opponent_df["opponent_fed"] = opponent_df["opponent_fed"].astype(str).str.strip()

    opponent_df = opponent_df[
        opponent_df["opponent_fed"].str.match(r"^[A-Z]{3}$", na=False)
    ].copy()

    if opponent_df.empty:
        return pd.DataFrame()

    opponent_df["opponent_title"] = (
        opponent_df[["opponent_title_1", "opponent_title_2"]]
        .astype(str)
        .replace("nan", "", regex=False)
        .agg(" ".join, axis=1)
        .str.strip()
    )

    opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan).infer_objects(copy=False)

    opponent_df["rating_difference_over_400_flag"] = (
        opponent_df["opponent_rating_raw"]
        .astype(str)
        .str.contains(r"\*", regex=True, na=False)
    )

    opponent_df["opponent_rating"] = (
        opponent_df["opponent_rating_raw"]
        .astype(str)
        .str.replace("*", "", regex=False)
        .str.strip()
    )

    opponent_df["opponent_rating"] = pd.to_numeric(
        opponent_df["opponent_rating"],
        errors="coerce"
    )

    for col in ["score", "n", "chg", "k", "k_chg"]:
        opponent_df[col] = pd.to_numeric(opponent_df[col], errors="coerce")

    opponent_df["fide_id"] = str(fide_id)
    opponent_df["period"] = period
    opponent_df["rating_type"] = rating_type

    opponent_df["tournament_name"] = event_meta.get("tournament_name")
    opponent_df["event_location_fed"] = event_meta.get("event_location_fed")
    opponent_df["event_start_date"] = event_meta.get("event_start_date")
    opponent_df["event_end_date"] = event_meta.get("event_end_date")

    opponent_df["event_rc"] = event_rc
    opponent_df["event_ro"] = event_ro
    opponent_df["event_score"] = event_score
    opponent_df["event_games"] = event_games
    opponent_df["event_chg"] = event_chg
    opponent_df["event_k"] = event_k
    opponent_df["event_k_chg"] = event_kchg

    keep_cols = [
        "fide_id", "period", "rating_type",
        "tournament_name", "event_location_fed",
        "event_start_date", "event_end_date",
        "event_rc", "event_ro", "event_score", "event_games",
        "event_chg", "event_k", "event_k_chg",
        "opponent_name", "opponent_title", "opponent_rating",
        "rating_difference_over_400_flag",
        "opponent_fed", "score", "n", "chg", "k", "k_chg"
    ]

    return opponent_df[keep_cols].reset_index(drop=True)

## 7. Save extraction results

This section saves the combined calculation-page dataset for later feature engineering, EDA, and modelling.


#### 5. Function to process one player-month

In [24]:
# ============================================================
# FIDE calculation-page extraction workflow
# Purpose:
# - This cell is part of the calculation-page extraction or validation process.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

async def extract_player_month(page, fide_id, period_str, rating_type=0, save_html=False):
    url = (
        "https://ratings.fide.com/calculations.phtml"
        f"?id_number={fide_id}&period={period_str}&rating={rating_type}"
    )

    await page.goto(url, wait_until="networkidle", timeout=60000)
    await page.wait_for_timeout(2000)

    html = await page.content()

    if save_html:
        player_html_dir = raw_html_dir / str(fide_id)
        player_html_dir.mkdir(parents=True, exist_ok=True)

        html_path = player_html_dir / f"{fide_id}_{period_str}_standard.html"
        html_path.write_text(html, encoding="utf-8")

    soup = BeautifulSoup(html, "html.parser")

    text_lines = [
        line.strip()
        for line in soup.get_text("\n", strip=True).split("\n")
        if line.strip()
    ]

    try:
        tables = pd.read_html(StringIO(html))
    except ValueError:
        tables = []

    events = extract_event_metadata_from_text(text_lines)

    clean_tables = []

    for i, table in enumerate(tables):
        event_meta = events[i] if i < len(events) else {}

        clean_one = clean_fide_calculation_table(
            table=table,
            event_meta=event_meta,
            fide_id=fide_id,
            period=period_str,
            rating_type=rating_type
        )

        if not clean_one.empty:
            clean_tables.append(clean_one)

    if clean_tables:
        clean_df = pd.concat(clean_tables, ignore_index=True)
    else:
        clean_df = pd.DataFrame()

    status = {
        "fide_id": str(fide_id),
        "period": period_str,
        "rating_type": rating_type,
        "url": url,
        "tables_found": len(tables),
        "events_found": len(events),
        "clean_rows": clean_df.shape[0],
        "status": "data_found" if clean_df.shape[0] > 0 else "no_data_or_no_clean_rows"
    }

    return clean_df, status

#### 6. Run for all players with resume support

Start with a small test first, for example 10 players.

## 8. Validation and summary checks

This section checks counts such as number of players with calculation data, number of opponent-game records, and sample rows to confirm the extraction worked.


In [25]:
# ============================================================
# FIDE calculation-page extraction workflow
# Purpose:
# - This cell is part of the calculation-page extraction or validation process.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

test_player_ids = player_ids[:2]
print(test_player_ids)

['558016627', '531075452']


In [26]:
# ============================================================
# FIDE calculation-page extraction workflow
# Purpose:
# - This cell is part of the calculation-page extraction or validation process.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

async def run_extraction_for_players(
    player_ids_to_run,
    periods,
    rating_type=0,
    save_html=False,
    delay_ms=1500
):
    all_status = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for idx, fide_id in enumerate(player_ids_to_run, start=1):
            print(f"\n===== Player {idx}/{len(player_ids_to_run)}: {fide_id} =====")

            player_clean_parts = []

            for period in periods:
                period_str = period.strftime("%Y-%m-%d")

                output_file = clean_dir / f"{fide_id}_{period_str}_standard_clean.csv"

                # Resume support: skip if already processed
                if output_file.exists():
                    print("Skipping existing:", output_file.name)

                    try:
                        existing_df = pd.read_csv(output_file)
                        clean_rows = existing_df.shape[0]
                    except Exception:
                        clean_rows = None

                    all_status.append({
                        "fide_id": str(fide_id),
                        "period": period_str,
                        "rating_type": rating_type,
                        "status": "skipped_existing",
                        "clean_rows": clean_rows
                    })
                    continue

                print("Processing:", fide_id, period_str)

                try:
                    clean_df, status = await extract_player_month(
                        page=page,
                        fide_id=fide_id,
                        period_str=period_str,
                        rating_type=rating_type,
                        save_html=save_html
                    )

                    # Save even if empty, so resume knows it was processed
                    clean_df.to_csv(output_file, index=False)

                    all_status.append(status)

                    if not clean_df.empty:
                        player_clean_parts.append(clean_df)
                        print("Rows:", clean_df.shape[0])
                    else:
                        print("No rows.")

                except Exception as e:
                    print("Failed:", fide_id, period_str, repr(e))

                    all_status.append({
                        "fide_id": str(fide_id),
                        "period": period_str,
                        "rating_type": rating_type,
                        "status": "failed",
                        "clean_rows": 0,
                        "error": repr(e)
                    })

                await page.wait_for_timeout(delay_ms)

            # Save player-level combined file
            if player_clean_parts:
                player_all = pd.concat(player_clean_parts, ignore_index=True)
                player_output = clean_dir / f"{fide_id}_all_months_standard_clean.csv"
                player_all.to_csv(player_output, index=False)
                print("Saved player file:", player_output, "Rows:", player_all.shape[0])

            # Save status after every player
            status_df = pd.DataFrame(all_status)
            status_path = log_dir / "fide_calculation_extraction_status.csv"
            status_df.to_csv(status_path, index=False)

        await browser.close()

    return pd.DataFrame(all_status)

In [27]:
# ============================================================
# FIDE calculation-page extraction workflow
# Purpose:
# - This cell is part of the calculation-page extraction or validation process.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

status_test = await run_extraction_for_players(
    player_ids_to_run=test_player_ids,
    periods=periods,
    rating_type=0,
    save_html=False,
    delay_ms=1500
)

display(status_test)


===== Player 1/2: 558016627 =====
Skipping existing: 558016627_2024-05-01_standard_clean.csv
Skipping existing: 558016627_2024-06-01_standard_clean.csv
Skipping existing: 558016627_2024-07-01_standard_clean.csv
Skipping existing: 558016627_2024-08-01_standard_clean.csv
Skipping existing: 558016627_2024-09-01_standard_clean.csv
Skipping existing: 558016627_2024-10-01_standard_clean.csv
Skipping existing: 558016627_2024-11-01_standard_clean.csv
Skipping existing: 558016627_2024-12-01_standard_clean.csv
Skipping existing: 558016627_2025-01-01_standard_clean.csv
Skipping existing: 558016627_2025-02-01_standard_clean.csv
Skipping existing: 558016627_2025-03-01_standard_clean.csv
Skipping existing: 558016627_2025-04-01_standard_clean.csv

===== Player 2/2: 531075452 =====
Skipping existing: 531075452_2024-05-01_standard_clean.csv
Skipping existing: 531075452_2024-06-01_standard_clean.csv
Skipping existing: 531075452_2024-07-01_standard_clean.csv
Skipping existing: 531075452_2024-08-01_stand

,fide_id,period,rating_type,status,clean_rows
0,558016627,2024-05-01,0,skipped_existing,NaN
1,558016627,2024-06-01,0,skipped_existing,NaN
2,558016627,2024-07-01,0,skipped_existing,NaN
3,558016627,2024-08-01,0,skipped_existing,NaN
4,558016627,2024-09-01,0,skipped_existing,NaN
5,558016627,2024-10-01,0,skipped_existing,NaN
6,558016627,2024-11-01,0,skipped_existing,NaN
7,558016627,2024-12-01,0,skipped_existing,NaN
8,558016627,2025-01-01,0,skipped_existing,NaN
9,558016627,2025-02-01,0,skipped_existing,NaN


In [42]:
# ============================================================
# FIDE calculation-page extraction workflow
# Purpose:
# - This cell is part of the calculation-page extraction or validation process.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

status_test.tail()

,fide_id,period,rating_type,status,clean_rows
19,531075452,2024-12-01,0,skipped_existing,NaN
20,531075452,2025-01-01,0,skipped_existing,NaN
21,531075452,2025-02-01,0,skipped_existing,6.0
22,531075452,2025-03-01,0,skipped_existing,NaN
23,531075452,2025-04-01,0,skipped_existing,NaN


In [ ]:
#7. If test works, run all 1,000 players

In [44]:
# ============================================================
# FIDE calculation-page extraction workflow
# Purpose:
# - This cell is part of the calculation-page extraction or validation process.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

status_all = await run_extraction_for_players(
    player_ids_to_run=player_ids,
    periods=periods,
    rating_type=0,
    save_html=False,
    delay_ms=2000
)

display(status_all)


===== Player 1/1000: 558016627 =====
Skipping existing: 558016627_2024-05-01_standard_clean.csv
Skipping existing: 558016627_2024-06-01_standard_clean.csv
Skipping existing: 558016627_2024-07-01_standard_clean.csv
Skipping existing: 558016627_2024-08-01_standard_clean.csv
Skipping existing: 558016627_2024-09-01_standard_clean.csv
Skipping existing: 558016627_2024-10-01_standard_clean.csv
Skipping existing: 558016627_2024-11-01_standard_clean.csv
Skipping existing: 558016627_2024-12-01_standard_clean.csv
Skipping existing: 558016627_2025-01-01_standard_clean.csv
Skipping existing: 558016627_2025-02-01_standard_clean.csv
Skipping existing: 558016627_2025-03-01_standard_clean.csv
Skipping existing: 558016627_2025-04-01_standard_clean.csv

===== Player 2/1000: 531075452 =====
Skipping existing: 531075452_2024-05-01_standard_clean.csv
Skipping existing: 531075452_2024-06-01_standard_clean.csv
Skipping existing: 531075452_2024-07-01_standard_clean.csv
Skipping existing: 531075452_2024-08-01

/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_49366/3931951331.py:90: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan).infer_objects(copy=False)


Rows: 5
Processing: 88128199 2024-07-01
No rows.
Processing: 88128199 2024-08-01
No rows.
Processing: 88128199 2024-09-01
No rows.
Processing: 88128199 2024-10-01
No rows.
Processing: 88128199 2024-11-01
No rows.
Processing: 88128199 2024-12-01
No rows.
Processing: 88128199 2025-01-01


/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_49366/3931951331.py:90: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan).infer_objects(copy=False)


Rows: 6
Processing: 88128199 2025-02-01
No rows.
Processing: 88128199 2025-03-01
No rows.
Processing: 88128199 2025-04-01


...

/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_49366/3931951331.py:90: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan).infer_objects(copy=False)


Rows: 6
Processing: 537026658 2025-01-01
No rows.
Processing: 537026658 2025-02-01


/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_49366/3931951331.py:90: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  opponent_df["opponent_title"] = opponent_df["opponent_title"].replace("", np.nan).infer_objects(copy=False)


Rows: 2
Processing: 537026658 2025-03-01
No rows.
Processing: 537026658 2025-04-01
No rows.
Saved player file: /Users/arunkumar/Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess/data/raw/fide_history/fide_calculation_pages/clean_player_months/537026658_all_months_standard_clean.csv Rows: 8

===== Player 998/1000: 547063394 =====
Processing: 547063394 2024-05-01
No rows.
Processing: 547063394 2024-06-01
No rows.
Processing: 547063394 2024-07-01
No rows.
Processing: 547063394 2024-08-01
No rows.
Processing: 547063394 2024-09-01
No rows.
Processing: 547063394 2024-10-01
No rows.
Processing: 547063394 2024-11-01
No rows.
Processing: 547063394 2024-12-01
No rows.
Processing: 547063394 2025-01-01
No rows.
Processing: 547063394 2025-02-01
No rows.
Processing: 547063394 2025-03-01
No rows.
Processing: 547063394 2025-04-01
No rows.

===== Player 999/1000: 531013880 =====
Processing: 531013880 2024-05-01
No rows.
Processing: 531013880 2024-06-01
No rows.

,fide_id,period,rating_type,status,clean_rows,url,tables_found,events_found
0,558016627,2024-05-01,0,skipped_existing,NaN,NaN,NaN,NaN
1,558016627,2024-06-01,0,skipped_existing,NaN,NaN,NaN,NaN
2,558016627,2024-07-01,0,skipped_existing,NaN,NaN,NaN,NaN
3,558016627,2024-08-01,0,skipped_existing,NaN,NaN,NaN,NaN
4,558016627,2024-09-01,0,skipped_existing,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
11995,547020970,2024-12-01,0,no_data_or_no_clean_rows,0.0,https://ratings.fide.com/calculations.phtml?id_number=547020970&period=2024-...,0.0,0.0
11996,547020970,2025-01-01,0,no_data_or_no_clean_rows,0.0,https://ratings.fide.com/calculations.phtml?id_number=547020970&period=2025-...,0.0,0.0
11997,547020970,2025-02-01,0,no_data_or_no_clean_rows,0.0,https://ratings.fide.com/calculations.phtml?id_number=547020970&period=2025-...,0.0,0.0
11998,547020970,2025-03-01,0,no_data_or_no_clean_rows,0.0,https://ratings.fide.com/calculations.phtml?id_number=547020970&period=2025-...,0.0,0.0


#### 8. Combine all extracted player-month files
After extraction, combine all non-empty files.

In [55]:
#combine from both folders.
# Set both folders:

from pathlib import Path
import pandas as pd
from pandas.errors import EmptyDataError

#clean_dir_1 = Path.home() / "Downloads" / "fide_history" / "fide_calculation_pages" / "clean_player_months"

clean_dir_1 = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw" / "fide_history" / "fide_calculation_pages" / "clean_player_months" 


# Change this to your second parallel process folder
#clean_dir_2 = Path.home() / "Downloads" / "fide_history" / "fide_calculation_pages_1" / "clean_player_months_1"
#clean_dir_2 = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "interim"
clean_dir_2 = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw" / "fide_history" / "fide_calculation_pages_1" / "clean_player_months_1" 


all_clean_files = sorted(list(clean_dir_1.glob("*_standard_clean.csv")) + 
                         list(clean_dir_2.glob("*_standard_clean.csv")))

print("Total files from both folders:", len(all_clean_files))

Total files from both folders: 15153


In [57]:
# Now combine safely and remove duplicate player-month files
file_rows = []

for f in all_clean_files:
    parts = f.name.split("_")
    fide_id = parts[0]
    period = parts[1]
    
    file_rows.append({
        "fide_id": fide_id,
        "period": period,
        "file_path": f,
        "file_name": f.name,
        "file_size": f.stat().st_size,
        "modified_time": f.stat().st_mtime
    })

file_index = pd.DataFrame(file_rows)

# If same player-month exists in both folders, keep the larger file;
# if same size, keep the most recently modified.
file_index = file_index.sort_values(
    ["fide_id", "period", "file_size", "modified_time"],
    ascending=[True, True, False, False]
)

file_index_dedup = file_index.drop_duplicates(
    subset=["fide_id", "period"],
    keep="first"
).copy()

print("Raw files:", file_index.shape[0])
print("Unique player-month files:", file_index_dedup.shape[0])

Raw files: 15153
Unique player-month files: 12628


In [59]:
# Check duplicate count

duplicates = file_index[file_index.duplicated(["fide_id", "period"], keep=False)]

print("Duplicate player-month files:", duplicates.shape[0])
display(duplicates.head(50))

Duplicate player-month files: 5050


,fide_id,period,file_path,file_name,file_size,modified_time
26,25100688,2024-05-01,/Users/arunkumar/Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_Tal...,25100688_2024-05-01_standard_clean.csv,1,1.780488e+09
12628,25100688,2024-05-01,/Users/arunkumar/Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_Tal...,25100688_2024-05-01_standard_clean.csv,1,1.778230e+09
27,25100688,2024-06-01,/Users/arunkumar/Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_Tal...,25100688_2024-06-01_standard_clean.csv,648,1.780488e+09
12629,25100688,2024-06-01,/Users/arunkumar/Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_Tal...,25100688_2024-06-01_standard_clean.csv,648,1.778230e+09
28,25100688,2024-07-01,/Users/arunkumar/Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_Tal...,25100688_2024-07-01_standard_clean.csv,1,1.780488e+09
12630,25100688,2024-07-01,/Users/arunkumar/Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_Tal...,25100688_2024-07-01_standard_clean.csv,1,1.778230e+09
29,25100688,2024-08-01,/Users/arunkumar/Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_Tal...,25100688_2024-08-01_standard_clean.csv,1,1.780488e+09
12631,25100688,2024-08-01,/Users/arunkumar/Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_Tal...,25100688_2024-08-01_standard_clean.csv,1,1.778230e+09
30,25100688,2024-09-01,/Users/arunkumar/Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_Tal...,25100688_2024-09-01_standard_clean.csv,1045,1.780488e+09
12632,25100688,2024-09-01,/Users/arunkumar/Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_Tal...,25100688_2024-09-01_standard_clean.csv,1045,1.778230e+09


In [61]:
# Now read only deduplicated files

combined_parts = []
empty_files = []
failed_files = []

for f in file_index_dedup["file_path"]:
    try:
        if f.stat().st_size == 0:
            empty_files.append(str(f))
            continue

        df = pd.read_csv(f)

        if df.empty:
            empty_files.append(str(f))
            continue

        combined_parts.append(df)

    except EmptyDataError:
        empty_files.append(str(f))

    except Exception as e:
        failed_files.append({
            "file": str(f),
            "error": repr(e)
        })

print("Unique files to read:", len(file_index_dedup))
print("Non-empty data files:", len(combined_parts))
print("Empty/no-data files:", len(empty_files))
print("Failed files:", len(failed_files))

if combined_parts:
    fide_calc_all_players = pd.concat(combined_parts, ignore_index=True)
else:
    fide_calc_all_players = pd.DataFrame()

print("Combined shape:", fide_calc_all_players.shape)
display(fide_calc_all_players.head(20))

Unique files to read: 12628
Non-empty data files: 2459
Empty/no-data files: 10169
Failed files: 0
Combined shape: (24204, 24)


,fide_id,period,rating_type,tournament_name,event_location_fed,event_start_date,event_end_date,event_rc,event_ro,event_score,event_games,event_chg,event_k,event_k_chg,opponent_name,opponent_title,opponent_rating,rating_difference_over_400_flag,opponent_fed,score,n,chg,k,k_chg
0,14350670,2024-05-01,0,Tamil Nadu State Women Chess Championship 2024,Kovilpatti IND,2024-04-24,2024-04-28,1577.0,1426.0,1.0,4,-0.21,40,-8.4,"Ananya, Manisundaram",NaN,1621.0,False,IND,0.0,1,-0.25,40.0,-10.0
1,14350670,2024-05-01,0,Tamil Nadu State Women Chess Championship 2024,Kovilpatti IND,2024-04-24,2024-04-28,1577.0,1426.0,1.0,4,-0.21,40,-8.4,"Lakshanaa, R",NaN,1633.0,False,IND,0.0,1,-0.23,40.0,-9.2
2,14350670,2024-05-01,0,Tamil Nadu State Women Chess Championship 2024,Kovilpatti IND,2024-04-24,2024-04-28,1577.0,1426.0,1.0,4,-0.21,40,-8.4,"Sandhana, D",NaN,1531.0,False,IND,0.0,1,-0.36,40.0,-14.4
3,14350670,2024-05-01,0,Tamil Nadu State Women Chess Championship 2024,Kovilpatti IND,2024-04-24,2024-04-28,1577.0,1426.0,1.0,4,-0.21,40,-8.4,"Rithika, M",NaN,1522.0,False,IND,1.0,1,0.63,40.0,25.2
4,14350670,2024-06-01,0,Tamil Nadu State Under 17 Girls FIDE Rated Chess Championship-2024,Arupukottai IND,2024-05-01,2024-05-05,1530.0,1418.0,2.0,6,-0.20,40,-8.0,"Shanmathi, Sree S",NaN,1818.0,True,IND,0.0,1,-0.08,40.0,-3.2
5,14350670,2024-06-01,0,Tamil Nadu State Under 17 Girls FIDE Rated Chess Championship-2024,Arupukottai IND,2024-05-01,2024-05-05,1530.0,1418.0,2.0,6,-0.20,40,-8.0,"Janani, J",NaN,1536.0,False,IND,0.0,1,-0.34,40.0,-13.6
6,14350670,2024-06-01,0,Tamil Nadu State Under 17 Girls FIDE Rated Chess Championship-2024,Arupukottai IND,2024-05-01,2024-05-05,1530.0,1418.0,2.0,6,-0.20,40,-8.0,"Madhavi, Sri V",NaN,1480.0,False,IND,0.5,1,0.09,40.0,3.6
7,14350670,2024-06-01,0,Tamil Nadu State Under 17 Girls FIDE Rated Chess Championship-2024,Arupukottai IND,2024-05-01,2024-05-05,1530.0,1418.0,2.0,6,-0.20,40,-8.0,"Anusri, S",NaN,1459.0,False,IND,0.0,1,-0.44,40.0,-17.6
8,14350670,2024-06-01,0,Tamil Nadu State Under 17 Girls FIDE Rated Chess Championship-2024,Arupukottai IND,2024-05-01,2024-05-05,1530.0,1418.0,2.0,6,-0.20,40,-8.0,"Keerthana, V",NaN,1453.0,False,IND,1.0,1,0.55,40.0,22.0
9,14350670,2024-06-01,0,Tamil Nadu State Under 17 Girls FIDE Rated Chess Championship-2024,Arupukottai IND,2024-05-01,2024-05-05,1530.0,1418.0,2.0,6,-0.20,40,-8.0,"Dhashya, Fiance S",NaN,1434.0,False,IND,0.5,1,0.02,40.0,0.8


In [63]:
#Save combined result
combined_output = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw" / "fide_history" / "fide_calculations_all_players_standard_clean_combined.csv"

fide_calc_all_players.to_csv(combined_output, index=False)

print("Saved:", combined_output)

Saved: /Users/arunkumar/Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess/data/raw/fide_history/fide_calculations_all_players_standard_clean_combined.csv


In [65]:
# Final coverage check
expected_ids = set(player_sample["ID Number"].astype(str).str.strip())
processed_ids = set(file_index_dedup["fide_id"].astype(str).str.strip())

print("Expected sample players:", len(expected_ids))
print("Processed players:", len(processed_ids))
print("Missing players:", len(expected_ids - processed_ids))

missing_players = sorted(expected_ids - processed_ids)
print("Missing player IDs:", missing_players[:100])

Expected sample players: 1000
Processed players: 1000
Missing players: 0
Missing player IDs: []


In [67]:
#And month coverage
expected_months = [p.strftime("%Y-%m-%d") for p in periods]
expected_count_per_player = len(expected_months)

coverage_by_player = (
    file_index_dedup
    .groupby("fide_id", as_index=False)
    .agg(months_processed=("period", "nunique"))
)

incomplete_players = coverage_by_player[
    coverage_by_player["months_processed"] < expected_count_per_player
].copy()

print("Expected months per player:", expected_count_per_player)
print("Incomplete players:", incomplete_players.shape[0])
display(incomplete_players.head(50))

Expected months per player: 12
Incomplete players: 0


,fide_id,months_processed


In [69]:
# ============================================================
# Code cell 39: FIDE calculation-page extraction workflow
# Purpose:
# - This cell is part of the calculation-page extraction or validation process.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

print("Rows:", fide_calc_all_players.shape[0])
print("Columns:", fide_calc_all_players.shape[1])
print("Players with opponent data:", fide_calc_all_players["fide_id"].nunique())
print("Unique tournaments:", fide_calc_all_players["tournament_name"].nunique())
print("Periods covered:", fide_calc_all_players["period"].min(), "to", fide_calc_all_players["period"].max())

Rows: 24204
Columns: 24
Players with opponent data: 629
Unique tournaments: 368
Periods covered: 2024-05-01 to 2025-04-01


In [71]:
# ============================================================
# Code cell 40: FIDE calculation-page extraction workflow
# Purpose:
# - This cell is part of the calculation-page extraction or validation process.
# - Code logic has not been changed; only explanatory comments were added.
# ============================================================

sample_ids = set(player_sample["ID Number"].astype(str).str.strip())
calc_ids = set(fide_calc_all_players["fide_id"].astype(str).str.strip())

print("Sample players:", len(sample_ids))
print("Players with calculation data:", len(calc_ids))
print("Calculation players not in sample:", len(calc_ids - sample_ids))
print("Sample players without calculation data:", len(sample_ids - calc_ids))

Sample players: 1000
Players with calculation data: 629
Calculation players not in sample: 0
Sample players without calculation data: 371


### Observations:
* Fide calcualtion data is now complete
* The 629 players have detailed opponent/tournament data from FIDE calculation pages.
* The 371 players without calculation data likely had one of these cases:
     * No Standard-rated games in the feature window.
     * They were present in rating lists but did not have monthly calculation details.
     * They had activity captured in monthly Gms, but FIDE calculation page did not expose tables for those months.
     * Their games may be outside Standard format if we only used rating_type = 0.

This is still useful because the main dataset remains the FIDE monthly rating panel, and the calculation-page data becomes an enrichment layer. This aligns well with the capstone’s planned use of FIDE rating history as the core dataset and tournament/opponent variables as enrichment features.